In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine, text

import pandas as pd
from shapely.ops import unary_union
import os

import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.features import geometry_mask
import numpy as np
from scipy import ndimage

In [ ]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    
def ensure_same_crs(gdf, raster):
    """
    Ensure GeoDataFrame CRS matches raster CRS.
    Reprojects vector if necessary.
    """
    if gdf.crs != raster.crs:
        print("Reprojecting vector to raster CRS...")
        gdf = gdf.to_crs(raster.crs)
    else:
        print("CRS already matches.")

    return gdf

def clip_raster_with_gdf(raster_path, gdf):
    """
    Clip raster using GeoDataFrame geometry.
    """
    with rasterio.open(raster_path) as src:
        gdf = ensure_same_crs(gdf, src)

        geoms = gdf.geometry.values

        clipped, transform = mask(
            src,
            geoms,
            crop=True
        )

        profile = src.profile.copy()
        nodata = src.nodata

    arr = clipped[0]

    return arr, transform, profile, nodata, geoms

def create_polygon_mask(shape, transform, geometries):
    """
    Raster mask of polygon area.
    """
    poly_mask = geometry_mask(
        geometries,
        transform=transform,
        invert=True,
        out_shape=shape
    )

    return poly_mask

def nearest_neighbor_fill(arr, nodata):
    """
    Fill nodata pixels using nearest neighbour.
    """

    if nodata is None:
        missing = np.isnan(arr)
    else:
        missing = (arr == nodata) | np.isnan(arr)

    valid_mask = ~missing

    print(f"Missing pixels: {missing.sum()}")

    indices = ndimage.distance_transform_edt(
        missing,
        return_distances=False,
        return_indices=True
    )

    filled = arr[tuple(indices)]

    return filled

def constrain_to_polygon(filled, polygon_mask, nodata):
    """
    Remove values outside territory.
    """
    filled[~polygon_mask] = nodata
    return filled

def save_raster(output_path, array, profile, transform, nodata):
    """
    Save raster to disk.
    """

    profile.update({
        "height": array.shape[0],
        "width": array.shape[1],
        "transform": transform,
        "count": 1,
        "nodata": nodata
    })

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(array, 1)

    print("Saved:", output_path)

In [ ]:
"""Load the data"""
# Load the GADM
gdf_gadm = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.100',
    port=5555,
    table_name='administrative_units_un_gadm_level0',
    schema='public'
)

src = rasterio.open("raster.tif")

In [ ]:
print("Clipping raster...")
arr, transform, profile, nodata, geoms = clip_raster_with_gdf(raster_path, gdf)

print("Creating polygon mask...")
poly_mask = create_polygon_mask(arr.shape, transform, geoms)

print("Filling gaps (nearest neighbour)...")
filled = nearest_neighbor_fill(arr, nodata)

print("Constraining to territory...")
filled = constrain_to_polygon(
    filled,
    poly_mask,
    nodata
)

print("Saving result...")
save_raster(
    output_path,
    filled,
    profile,
    transform,
    nodata
)

print("Done ✅")